# Business Sales Performance Analytics
### Future Interns - Data Science & Analytics Internship
**Name:** Pawan Kumar Jatav  
**Track:** Data Science & Analytics  
**Task:** Task 1 - Business Sales Performance Analytics

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# using seaborn whitegrid, looks cleaner
plt.style.use('seaborn-v0_8-whitegrid')

print('libraries loaded')

## Creating the Dataset
Since no dataset was provided, I created a realistic sales dataset manually using numpy and pandas. It has 1000 transactions covering the full year 2023.

In [ ]:
np.random.seed(42)
n = 1000

products = ['Laptop', 'Mobile Phone', 'Tablet', 'Headphones', 'Smart Watch',
            'Camera', 'Keyboard', 'Mouse', 'Monitor', 'Printer']

categories = {
    'Laptop': 'Electronics', 'Mobile Phone': 'Electronics',
    'Tablet': 'Electronics', 'Headphones': 'Accessories',
    'Smart Watch': 'Wearables', 'Camera': 'Electronics',
    'Keyboard': 'Accessories', 'Mouse': 'Accessories',
    'Monitor': 'Electronics', 'Printer': 'Office Equipment'
}

base_prices = {
    'Laptop': 55000, 'Mobile Phone': 20000, 'Tablet': 25000,
    'Headphones': 3000, 'Smart Watch': 8000, 'Camera': 35000,
    'Keyboard': 1500, 'Mouse': 800, 'Monitor': 15000, 'Printer': 12000
}

regions = ['North', 'South', 'East', 'West', 'Central']

dates = pd.date_range(start='2023-01-01', end='2023-12-31', periods=n)
dates = np.sort(np.random.choice(dates, n, replace=False))

product_list = np.random.choice(products, n, p=[0.15, 0.20, 0.10, 0.12, 0.08,
                                                  0.07, 0.10, 0.08, 0.06, 0.04])

df = pd.DataFrame({
    'Order_ID': ['ORD' + str(i).zfill(4) for i in range(1, n+1)],
    'Date': dates,
    'Product': product_list,
    'Category': [categories[p] for p in product_list],
    'Region': np.random.choice(regions, n),
    'Quantity': np.random.randint(1, 6, n),
    'Unit_Price': [base_prices[p] * np.random.uniform(0.9, 1.1) for p in product_list],
    'Discount_Pct': np.random.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.2, 0.2, 0.1, 0.1])
})

df['Gross_Revenue'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Revenue'] * df['Discount_Pct'] / 100
df['Net_Revenue'] = df['Gross_Revenue'] - df['Discount_Amount']
df['Month'] = df['Date'].dt.month_name()
df['Month_Num'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})

print('dataset ready, shape:', df.shape)
df.head()

## Basic Data Check

In [ ]:
print(df.info())
print('\nnull values:')
print(df.isnull().sum())

# no null values found, data looks clean

In [ ]:
df[['Quantity', 'Unit_Price', 'Net_Revenue']].describe().round(2)

## Overall Business KPIs

In [ ]:
total_revenue = df['Net_Revenue'].sum()
total_orders = df['Order_ID'].nunique()
total_units = df['Quantity'].sum()
avg_order_value = total_revenue / total_orders
total_discount = df['Discount_Amount'].sum()

print('Total Net Revenue     :', round(total_revenue, 2))
print('Total Orders          :', total_orders)
print('Total Units Sold      :', total_units)
print('Avg Order Value       :', round(avg_order_value, 2))
print('Total Discount Given  :', round(total_discount, 2))

In [ ]:
# visualizing KPIs as simple card-style chart
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Key Business KPIs - 2023', fontsize=14, fontweight='bold')

kpis = [
    ('Total Revenue', f'Rs {total_revenue/1e6:.1f}M', '#2ecc71'),
    ('Total Orders', f'{total_orders}', '#3498db'),
    ('Units Sold', f'{total_units}', '#9b59b6'),
    ('Avg Order Value', f'Rs {avg_order_value:,.0f}', '#e74c3c')
]

for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor(color)
    ax.text(0.5, 0.6, value, ha='center', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=11,
            color='white', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.savefig('kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Monthly Sales Trend

In [ ]:
monthly = df.groupby('Month_Num').agg(
    Revenue=('Net_Revenue', 'sum'),
    Orders=('Order_ID', 'count')
).reset_index()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['Month_Name'] = month_names

fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.bar(monthly['Month_Name'], monthly['Revenue']/1000, color='steelblue', alpha=0.75, label='Revenue (Rs K)')
ax1.set_ylabel('Net Revenue (Rs Thousands)')
ax1.set_xlabel('Month')

ax2 = ax1.twinx()
ax2.plot(monthly['Month_Name'], monthly['Orders'], color='red', marker='o', linewidth=2, label='Orders')
ax2.set_ylabel('Number of Orders', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Monthly Revenue and Order Volume - 2023', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

best_month = monthly.loc[monthly['Revenue'].idxmax(), 'Month_Name']
worst_month = monthly.loc[monthly['Revenue'].idxmin(), 'Month_Name']
print('Best month by revenue:', best_month)
print('Lowest revenue month:', worst_month)
# I noticed revenue is not consistent throughout the year, some months clearly perform better

## Product-wise Analysis

In [ ]:
product_perf = df.groupby('Product').agg(
    Revenue=('Net_Revenue', 'sum'),
    Orders=('Order_ID', 'count'),
    Units=('Quantity', 'sum')
).sort_values('Revenue', ascending=False).reset_index()

product_perf

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = sns.color_palette('Set2', len(product_perf))

axes[0].barh(product_perf['Product'], product_perf['Revenue']/1000, color=colors)
axes[0].set_xlabel('Net Revenue (Rs Thousands)')
axes[0].set_title('Revenue by Product')
axes[0].invert_yaxis()

axes[1].barh(product_perf['Product'], product_perf['Units'], color=colors)
axes[1].set_xlabel('Total Units Sold')
axes[1].set_title('Units Sold by Product')
axes[1].invert_yaxis()

plt.suptitle('Product Performance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('product_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Laptop and Mobile Phone are generating the highest revenue, which makes sense given their higher price points
# Mouse and Keyboard sell decent units but contribute less revenue because they are cheap products

## Category Performance

In [ ]:
cat_perf = df.groupby('Category').agg(
    Revenue=('Net_Revenue', 'sum'),
    Orders=('Order_ID', 'count')
).sort_values('Revenue', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(cat_perf['Revenue'], labels=cat_perf['Category'], autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('Set3', len(cat_perf)))
axes[0].set_title('Revenue Share by Category')

axes[1].bar(cat_perf['Category'], cat_perf['Orders'],
            color=sns.color_palette('Set3', len(cat_perf)))
axes[1].set_title('Orders by Category')
axes[1].set_ylabel('Number of Orders')

plt.suptitle('Category-wise Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Electronics dominates revenue by a big margin
# Accessories have more orders but much lower revenue per order

## Regional Performance

In [ ]:
region_perf = df.groupby('Region').agg(
    Revenue=('Net_Revenue', 'sum'),
    Orders=('Order_ID', 'count'),
    Avg_Order_Value=('Net_Revenue', 'mean')
).sort_values('Revenue', ascending=False).reset_index()

region_perf.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = sns.color_palette('coolwarm', len(region_perf))

axes[0].bar(region_perf['Region'], region_perf['Revenue']/1000, color=colors)
axes[0].set_title('Revenue by Region')
axes[0].set_ylabel('Revenue (Rs K)')

axes[1].bar(region_perf['Region'], region_perf['Orders'], color=colors)
axes[1].set_title('Orders by Region')
axes[1].set_ylabel('Orders')

axes[2].bar(region_perf['Region'], region_perf['Avg_Order_Value'], color=colors)
axes[2].set_title('Avg Order Value by Region')
axes[2].set_ylabel('Avg Value (Rs)')

plt.suptitle('Regional Performance - 2023', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Interesting - some regions have high orders but lower avg order value
# This suggests they buy more cheaper products like accessories

## Quarterly Comparison

In [ ]:
quarterly = df.groupby('Quarter').agg(
    Revenue=('Net_Revenue', 'sum'),
    Orders=('Order_ID', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(quarterly['Quarter'], quarterly['Revenue']/1000,
              color=['#3498db','#2ecc71','#e74c3c','#f39c12'], width=0.5)

for bar, orders in zip(bars, quarterly['Orders']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            f'{orders} orders', ha='center', fontsize=9)

ax.set_title('Quarterly Revenue Comparison - 2023', fontsize=13, fontweight='bold')
ax.set_xlabel('Quarter')
ax.set_ylabel('Net Revenue (Rs Thousands)')

plt.tight_layout()
plt.savefig('quarterly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Discount Impact on Sales

In [ ]:
discount_analysis = df.groupby('Discount_Pct').agg(
    Total_Orders=('Order_ID', 'count'),
    Total_Revenue=('Net_Revenue', 'sum')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(discount_analysis['Discount_Pct'].astype(str)+'%',
            discount_analysis['Total_Orders'], color='steelblue')
axes[0].set_title('Orders by Discount Level')
axes[0].set_xlabel('Discount %')
axes[0].set_ylabel('Orders')

axes[1].bar(discount_analysis['Discount_Pct'].astype(str)+'%',
            discount_analysis['Total_Revenue']/1000, color='coral')
axes[1].set_title('Revenue by Discount Level')
axes[1].set_xlabel('Discount %')
axes[1].set_ylabel('Revenue (Rs K)')

plt.suptitle('Discount Impact Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('discount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Most purchases happen at 0% discount which is good
# Discounts increased orders a bit but total revenue went down because of lower price
# So giving too much discount is not really helping the business

## Region x Product Revenue Heatmap

In [ ]:
pivot = df.pivot_table(values='Net_Revenue', index='Region',
                        columns='Product', aggfunc='sum') / 1000

plt.figure(figsize=(13, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, cbar_kws={'label': 'Revenue (Rs K)'})
plt.title('Revenue Heatmap: Region x Product (Rs Thousands)', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('heatmap_region_product.png', dpi=150, bbox_inches='tight')
plt.show()

# Laptop revenue is high across almost all regions
# Mouse and Keyboard show very low numbers everywhere as expected due to low price

## Summary and Observations

In [ ]:
top_product = product_perf.iloc[0]
top_region = region_perf.iloc[0]
top_category = cat_perf.iloc[0]
best_quarter = quarterly.loc[quarterly['Revenue'].idxmax()]

print('Key Findings from the Analysis:')
print()
print('1. Top Product:', top_product['Product'])
print('   Revenue:', round(top_product['Revenue'], 2), '| Units sold:', top_product['Units'])
print('   This product alone contributes the most to overall revenue.')
print()
print('2. Best Performing Region:', top_region['Region'])
print('   Revenue:', round(top_region['Revenue'], 2), '| Orders:', top_region['Orders'])
print()
print('3. Top Category:', top_category['Category'])
print('   Electronics is dominating because most high-value products fall here.')
print()
print('4. Best Quarter:', best_quarter['Quarter'])
print('   Revenue:', round(best_quarter['Revenue'], 2))
print()
print('5. Discount Observation:')
print('   Most orders placed without any discount. High discounts reduced revenue without proportional order increase.')
print()
print('Recommendations:')
print('- Focus more on Laptop and Mobile Phone inventory as demand is high')
print('- Invest in marketing for low-performing regions to increase reach')
print('- Avoid giving 15-20% discounts as they hurt revenue more than they help')
print('- Plan ahead for the best quarter with better stock management')
print('- Accessories category needs bundling strategy to increase avg order value')

## Saving the Dataset

In [ ]:
df.to_csv('sales_data.csv', index=False)
print('Dataset saved as sales_data.csv')
print('All charts saved as png files')
print('Ready to upload on GitHub repo: FUTURE_DS_01')